In [ ]:
%pip install -q google-genai pydantic

In [12]:
import json
import time
import pandas as pd

from google import genai
from google.genai import types
from pydantic import BaseModel,Field, ValidationError


In [ ]:
# Creating root_cause list
root_causes = [
    'forgot_or_no_remainder',
    'distance_or_transport',
    'cost_or_fee',
    'ittness_or_emergency',
    'work_or_schedule',
    'other'
] 

In [ ]:
# Creating a SCHEMA that will be given to gemini as response_schema
SCHEMA = {  
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"}, 
        "confidence": {"type": "number"},
        "summary": {"type": "string"}, 
        "recommended_action": {"type": "string"}
        },
    "required": [
        "root_cause", 
        "confidence", 
        "summary", 
        "recommended_action"
    ]
}

# Creating SYSTEM_TEXT, whcih will be given to gemini as system_instruction
SYSTEM_TEXT = '''
    You are a healthcare call-centre analyst. Your task is to identify why a patient missed an appointment.
    Return only valid JSON matching the provided schema
'''

In [ ]:
# Paste Your Gemini API Key Here
API_KEY = input("Paste your key: ")
client = genai.Client(api_key=API_KEY)

# Task 4.1

In [ ]:
# Defining a Function that sends one note at a time to gemini model and returns the appropriate response

def analyse_note(note_text):
    prompt = f'''
        Context: Provided are the call centre notes from a healthcare appointment system.
        which includes the Patients who missed the appointments, with the reason 

        Role: You are an expert healthcare operations analyst.

        Action: Read the note and determine:
        1: Root cause (a plain string - do not make it an enum)
        2. Confidence score between 0 and 1
        3. Short summary (at most 25 words )
        4. Recommendation action

        Allowed root_cause values: {','.join(root_causes)}

        Tone: Objective, concise, and professional.

        Format: Return ONLY JSON matching the supplied schema.

        Call-centre note: {note_text}
    '''
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',                      
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,

            response_mime_type="application/json", 
                response_schema=SCHEMA,

        thinking_config=types.ThinkingConfig(thinking_level="minimal")
        )
    )

    data = json.loads(response.text)

    return data

In [ ]:
# Reading the followup_notes CSV file
resp = pd.read_csv('data/followup_notes.csv')
resp.head()

,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."


In [ ]:
# running the function analyse_note for the first note_text
note_n001 = resp.loc[resp['note_id'] == 'N001', 'note_text'].iloc[0]
print(note_n001)
result = analyse_note(note_n001)

print(result)

Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
{'root_cause': 'forgot_or_no_remainder', 'confidence': 0.95, 'summary': 'Patient missed the appointment because she did not receive a reminder and forgot.', 'recommended_action': 'Enable appointment reminder notifications and assist patient in rescheduling for next week.'}


# Validate and process the 10 notes

In [ ]:
# Creating a pydantic model to validate the response 
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1) 
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


In [20]:
if result['root_cause'] in root_causes:
    print(f'{result['root_cause']} is in the root_causes allowed list')

forgot_or_no_remainder is in the root_causes allowed list


In [ ]:
# running the analyse_note function on the first 10 note_text, and validating the response.
# Finally saving the validated response to result list.

result = []


for i in range(10):
    raw_output = {}                                             # to save the response from the function
    raw_response = ""                                           # raw response 
    validation_status = "invalid"

    # parsing the followup_notes DataFrame
    note_id = resp['note_id'][i]
    appointment_id = resp['appointment_id'][i]
    note_text = resp['note_text'][i]

    # saving a response where the note_text  is None
    if (pd.isna(note_text)) or str(note_text).strip() == '':
        result.append({
            'note_id': note_id,
            'appointment_id': appointment_id, 
            'root_cause': 'other', 
            'confidence': 0.0, 
            'summary': 'No note recorded', 
            'recommended_action': 'Re-call patient', 
            'validation_status': 'skipped_blank', 
            'raw_response': ''
        })
        continue                                                # Skipping to next loop
    
    try:
        raw_output = analyse_note(note_text)                    # Function call

        raw_response = json.dumps(raw_output)
        insight = FollowupInsight(**raw_output)                 # Validating the result

        
        if insight.root_cause not in root_causes:               
            raise ValueError("Invalid root_cause")

        validation_status = 'valid'

        # Saving the result in the list
        result.append({
            'note_id': note_id,
            'appointment_id': appointment_id, 
            'root_cause': raw_output.get('root_cause',''), 
            'confidence': raw_output.get('confidence',''), 
            'summary': raw_output.get('summary',''), 
            'recommended_action': raw_output.get('recommended_action',''), 
            'validation_status' : validation_status, 
            'raw_response': raw_response
        })
        
    except Exception as e:                                      # Managing Exceptions if any
        if raw_response == '':
            raw_response = str(e)
        result.append({                                         # Saving the response with the exception
            'note_id': note_id,
            'appointment_id': appointment_id, 
            'root_cause': raw_output.get('root_cause',''), 
            'confidence': raw_output.get('confidence',''), 
            'summary': raw_output.get('summary',''), 
            'recommended_action': raw_output.get('recommended_action',''), 
            'validation_status' : validation_status, 
            'raw_response': raw_response
        })
    time.sleep(13)

In [ ]:
# Converting the result list to pandas DataFrame
analysed_df = pd.DataFrame(result)

analysed_df.head()

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_remainder,0.95,Patient missed the appointment because she did...,Enable alternative reminder methods and assist...,valid,"{""root_cause"": ""forgot_or_no_remainder"", ""conf..."
1,N002,A0897,distance_or_transport,0.95,Patient missed the appointment because they li...,Offer telehealth options or assist with arrang...,valid,"{""root_cause"": ""distance_or_transport"", ""confi..."
2,N003,A0526,cost_or_fee,0.95,Patient skipped the appointment because the co...,Provide information on available financial ass...,valid,"{""root_cause"": ""cost_or_fee"", ""confidence"": 0...."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,ittness_or_emergency,0.95,Patient missed the appointment due to a high f...,Reschedule the appointment once the patient ha...,valid,"{""root_cause"": ""ittness_or_emergency"", ""confid..."


In [ ]:
# Saving the DataFrame into notes_analysed CSV file
analysed_df.to_csv(
    "notes_analysed.csv",
    index=False
)

print("Saved notes_analysed.csv")

Saved notes_analysed.csv


# Task 4.3 Break it on purpose

In [ ]:
# Creating and Validating a bad reply bad_reply_1
bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot appointment.",
    "recommended_action": "Send reminder."
}

try:
    FollowupInsight(**bad_reply_1)

except ValidationError as e:
    print("BAD REPLY 1")
    print(e)

BAD REPLY 1
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [ ]:
# Creating a bad reply bad_reply_2

bad_reply_2 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.8,
    "summary": "Patient forgot appointment."
}

In [ ]:
# Validating a bad reply bad_reply_2
try:
    FollowupInsight(**bad_reply_2)

except ValidationError as e:
    print("BAD REPLY 2")
    print(e)

BAD REPLY 2
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'forgot_or...nt forgot appointment.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:
# Creating the simple prompt prompt_v1
prompt_v1 = f'''classify this note: 
{note_text}
'''

In [ ]:
# Creating the detailed prompt prompt_v2
prompt_v2 = f'''
    Context: Provided are the health care notes.

    Role: You are a healthcare operations analyst.

    Action: Identify the primary root cause.

    Allowed root_cause values:{root_causes}

        If uncertain, return other rather than guessing.

    Tone: Objective and concise.

    Format: Return JSON.

    Note: {note_text}
'''

Since we need **same configuration** for both the prompt, we can create a fuction with a set configurations that will run both of the prompt, instead I rewriting the configuration for the prompts. 

In [85]:
#Helper Function

def run_prompt(note_text,prompt):
    response = client.models.generate_content(
        model = 'gemini-3.5-flash-lite',
        contents = prompt,
        config = types.GenerateContentConfig(
            temperature = 0,
            max_output_tokens = 1000,

            response_mime_type='application/json',
            response_schema = SCHEMA,

            thinking_config = types.ThinkingConfig(thinkingLevel='minimal')
        )
    )

    return json.loads(response.text)

In [ ]:
# Result Table
result_table = []

In [ ]:
# Saving note_text of N007 in n007 and of N009 in n009 
n007 = resp.loc[resp['note_id'] == 'N007','note_text'].iloc[0]

n009 = resp.loc[resp['note_id'] == 'N009','note_text'].iloc[0]

n007,n009

('Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.',
 'Phone was switched off, could not reach the patient after three attempts.')

In [ ]:
# N007 + V1 prompt
r = run_prompt(n007,prompt_v1)

result_table.append({
    "note_id": 'N007',
    'prompt': 'V1',
    'root_causes': r.get('root_cause'),
    'allowed': r.get('root_cause') in root_causes
})

In [ ]:
#N007 + V2 prompt
r = run_prompt(n007,prompt_v2)

result_table.append({
    "note_id": 'N007',
    'prompt': 'V2',
    'root_causes': r.get('root_cause'),
    'allowed': r.get('root_cause') in root_causes
})

In [ ]:
# N009 + V1 prompt
r = run_prompt(n009,prompt_v1)

result_table.append({
    "note_id": 'N009',
    'prompt': 'V1',
    'root_causes': r.get('root_cause'),
    'allowed': r.get('root_cause') in root_causes
})

In [ ]:
# N009 + V2 prompt
r = run_prompt(n009,prompt_v2)

result_table.append({
    "note_id": 'N009',
    'prompt': 'V2',
    'root_causes': r.get('root_cause'),
    'allowed': r.get('root_cause') in root_causes
})

In [ ]:
# Comparing the result and Finding the best prompt style
comparison_df = pd.DataFrame(result_table)

comparison_df

,note_id,prompt,root_causes,allowed
0,N007,V1,Work schedule conflict due to factory shift ch...,False
1,N007,V2,work_or_schedule,True
2,N009,V1,Schedule conflict due to factory shift change ...,False
3,N009,V2,work_or_schedule,True


The __*V2 CRAFT*__ prompt would be deployed because it consistently constrains responses to the required categories and instructs the model to return "other" when evidence is insufficient. The V1 prompt is ambiguous and may produce labels outside the allowed `root_causes` list, increasing validation failures and reducing consistency. Since both prompts use identical model settings and schema, the improved reliability comes from **better prompt design rather than model configuration.**

In [ ]:
# Counting the total number of API calls
total_api_calls = 1 + 9 + 4

print("Total API Calls:", total_api_calls)

Total API Calls: 14


# Task 4.4 Recommendation

In [ ]:
# reading the CSV file appointment_clear
appointments = pd.read_csv('appointments_clean.csv')

In [ ]:
# Left merging of notes_analysed file with appointment_clear file on appointment_id 
merged = pd.merge(analysed_df, appointments, on= 'appointment_id',how= 'left' )

merged.head()

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,N001,A0872,forgot_or_no_remainder,0.95,Patient missed the appointment because she did...,Enable alternative reminder methods and assist...,valid,"{""root_cause"": ""forgot_or_no_remainder"", ""conf...",P144,C05,...,F,17.6,750,App,Consultation,No,1,No-Show,14,1
1,N002,A0897,distance_or_transport,0.95,Patient missed the appointment because they li...,Offer telehealth options or assist with arrang...,valid,"{""root_cause"": ""distance_or_transport"", ""confi...",P268,C07,...,M,17.2,1219,Phone,Follow-up,No,0,No-Show,5,1
2,N003,A0526,cost_or_fee,0.95,Patient skipped the appointment because the co...,Provide information on available financial ass...,valid,"{""root_cause"": ""cost_or_fee"", ""confidence"": 0....",P006,C04,...,F,5.3,800,App,Consultation,Yes,0,No-Show,34,1
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,,P449,C04,...,M,4.5,300,Phone,Follow-up,No,0,No-Show,40,1
4,N005,A0534,ittness_or_emergency,0.95,Patient missed the appointment due to a high f...,Reschedule the appointment once the patient ha...,valid,"{""root_cause"": ""ittness_or_emergency"", ""confid...",P400,C04,...,F,4.2,550,App,Follow-up,Yes,0,No-Show,26,1


**Cross Tab** on
root_cause vs remainder_sent

In [ ]:
cross_tab = pd.crosstab(
    merged['root_cause'],
    merged['reminder_sent']
)

print(cross_tab)

reminder_sent           No  Yes
root_cause                     
cost_or_fee              0    1
distance_or_transport    1    1
forgot_or_no_remainder   2    1
ittness_or_emergency     0    1
other                    1    0
work_or_schedule         1    1
